# NSC: Araya & Reyes (2019) en Python

Este notebook reproduce los experimentos del repositorio
[DaniSwi/ScriptDAG](https://github.com/DaniSwi/ScriptDAG), que implementa la Sección 3 de:

> Araya, I. & Reyes, V. (2019). *Enhancing interval constraint propagation by identifying and filtering n-ary subsystems.* Journal of Global Optimization, 74, 1–20.

Se comparan las tres estrategias de la Sección 3 del paper, con el mismo solver (branch & bound por intervalos, Newton intervalar y bisector SmearSumRel):

| Estrategia | Qué hace |
|---|---|
| **ARB** | HC4 sobre los árboles de expresión: **sin preprocesamiento** |
| **REF** | HC4 sobre el DAG, que fusiona subexpresiones comunes |
| **NSC** | DAG + filtrado del subsistema lineal de sumas con el conditioner `P` (§3.3.2) |

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Las secciones 1 a 4 tardan unos 2 minutos en total. La sección 5 corre un experimento chico en vivo (unos 6 minutos), y la 6 explica cómo correr el experimento completo, que tarda horas.

## 0. Preparar el entorno

Clona el repositorio. La única dependencia es `numpy`, que Colab ya trae.

In [ ]:
RAMA = "main"
!rm -rf ScriptDAG && git clone -q --depth 1 -b {RAMA} https://github.com/DaniSwi/ScriptDAG.git
%cd ScriptDAG
!python --version
!git log --oneline -1

## 1. Pruebas del repositorio

- `test_paper.py`: el preprocesamiento (DAG y matriz `A`) sobre el ejemplo (1) del paper.
- `test_nsc.py`: el conditioner reproduce exactamente la `P` de la §3.3.2, y NSC nunca elimina una solución (con soluciones fabricadas).
- `test_bb.py`: seno y coseno por intervalos, jacobiano de Newton, el solver sobre el ejemplo (1), el generador de instancias y el modo sin preprocesamiento.

Cada una debe terminar en `Todo OK.`

In [ ]:
!python test_paper.py | tail -1
!python test_nsc.py | tail -1
!python test_bb.py

## 2. Ejemplo (1) del paper: con y sin preprocesamiento

El sistema de la página 2 del paper, con dominios `[-1, 5]`. En la página 8 el paper reporta árboles de **325**, **83** y **33** nodos para las tres estrategias.

In [ ]:
from bb import comparar_bb
from exportator import construir
from test_nsc import EJEMPLO_1_TXT

with open("ejemplo1.txt", "w") as f:
    f.write(EJEMPLO_1_TXT)
print(EJEMPLO_1_TXT)

inst, sis, _ = construir("ejemplo1.txt")
resultados = comparar_bb(inst, sis, ruta="ejemplo1.txt")

paper = {"ARB": 325, "REF": 83, "NSC": 33}
print()
print(f"{'estrategia':10s} {'cajas aqui':>11s} {'paper':>7s}")
for r in resultados:
    print(f"{r.estrategia:10s} {r.cajas:11d} {paper[r.estrategia]:7d}")

## 3. Resultados guardados del experimento de la Sección 4.2

Instancias aleatorias con la misma estructura que las del paper (ecuaciones 6 y 7, n = 8, Q = 100, P de 8 a 16), 10 instancias por P y tipo, límite de 300 s por árbol. **Diferencia con el paper:** dominios `[-2, 2]` en vez de `[-1e8, 1e8]`, porque en Python el solver no termina con los originales.

Estos son los resultados que ya se corrieron (150 árboles por tipo, 50 por estrategia, en `experimentos/resultados_*.csv`). Primero, cuántas instancias termina cada estrategia dentro del límite:

In [ ]:
import csv
from collections import defaultdict

import experimento

filas = {t: list(csv.DictReader(open(f"experimentos/resultados_{t}.csv"))) for t in ("mult", "func")}

terminadas = defaultdict(int)
corridas = defaultdict(int)
for t, fs in filas.items():
    for r in fs:
        k = (t, int(r["P"]), r["estrategia"])
        corridas[k] += 1
        terminadas[k] += r["completo"] == "True"

for t in ("mult", "func"):
    print(f"--- {t}: instancias terminadas en 300 s ---")
    print(f"{'P':>3s} {'ARB':>7s} {'REF':>7s} {'NSC':>7s}")
    for P in (8, 10, 12, 14, 16):
        print(f"{P:3d} " + " ".join(f"{terminadas[(t, P, e)]:3d}/{corridas[(t, P, e)]:<3d}"
                                    for e in ("ARB", "REF", "NSC")))
    total = {e: sum(terminadas[(t, P, e)] for P in (8, 10, 12, 14, 16)) for e in ("ARB", "REF", "NSC")}
    print(f"tot {total['ARB']:3d}/50 {total['REF']:3d}/50 {total['NSC']:3d}/50\n")

Ganancia relativa promedio (Definición 3 del paper) en tiempo (`gt`) y en cajas (`gb`). Un valor mayor que 1 significa que la segunda estrategia gana. Cada comparación usa solo las instancias que **las dos** estrategias terminaron (`n`).

In [ ]:
for base, otra in experimento.COMPARACIONES:
    for t in ("mult", "func"):
        experimento.imprimir(experimento.resumir(filas[t], base, otra), base, otra)
        print()

## 4. Gráficos

A la izquierda, cuántas instancias termina cada estrategia (el efecto del preprocesamiento). A la derecha, la ganancia en cajas de NSC respecto del DAG solo, que es lo que muestran las Fig. 6 y 7 del paper.

In [ ]:
import matplotlib.pyplot as plt

Ps = [8, 10, 12, 14, 16]
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))

colores = {"ARB": "#9aa3a8", "REF": "#4f7cac", "NSC": "#0f6b66"}
for k, (t, estilo) in enumerate((("mult", None), ("func", "//"))):
    for j, e in enumerate(("ARB", "REF", "NSC")):
        ejes[0].bar([i + (j - 1) * 0.27 + (k - 0.5) * 0.12 for i in range(len(Ps))],
                    [terminadas[(t, P, e)] for P in Ps], width=0.12,
                    color=colores[e], hatch=estilo, edgecolor="white",
                    label=f"{e} ({t})")
ejes[0].set_xticks(range(len(Ps)), [str(P) for P in Ps])
ejes[0].set_xlabel("P (sumas en el sistema)")
ejes[0].set_ylabel("instancias terminadas (de 10)")
ejes[0].set_title("Con y sin preprocesamiento")
ejes[0].legend(fontsize=8, ncol=2)

for t, color in (("mult", "#0f6b66"), ("func", "#6b4fb3")):
    res = experimento.resumir(filas[t], "REF", "NSC")
    ejes[1].plot(Ps, [res[(t, P)]["gb"] for P in Ps], "o-", color=color, label=f"{t} (gb)")
    ejes[1].plot(Ps, [res[(t, P)]["gt"] for P in Ps], "s--", color=color, alpha=0.6, label=f"{t} (gt)")
ejes[1].axhline(1, color="gray", lw=1, ls=":")
ejes[1].set_xticks(Ps)
ejes[1].set_xlabel("P (sumas en el sistema)")
ejes[1].set_ylabel("ganancia de NSC respecto de REF")
ejes[1].set_title("NSC vs DAG solo (como Fig. 6 y 7)")
ejes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Experimento chico en vivo (opcional)

Corre las tres estrategias sobre 2 instancias nuevas por cada P = 8 y 10, con un límite de 60 s por árbol, en los 2 núcleos de Colab. Son 12 árboles y tarda como máximo unos 6 minutos. Las instancias se generan con semillas fijas, así que el resultado es reproducible. Con tan pocas instancias los números varían bastante; sirve para ver el procedimiento, no para sacar conclusiones.

In [ ]:
!python experimento.py --tipo mult --P 8 10 --instancias 2 --limite 60 --procesos 2 --carpeta exp_colab

## 6. Experimento completo

Para repetir el experimento completo de la sección 3 (10 instancias por P, 300 s por árbol, 150 árboles por tipo):

```
!python experimento.py --tipo mult --instancias 10 --limite 300 --carpeta exp_completo
!python experimento.py --tipo func --instancias 10 --limite 300 --carpeta exp_completo
```

Con los 2 núcleos de Colab cada tipo tarda varias horas, más que el límite de una sesión gratuita. Si la sesión se corta, se retoma agregando `--reanudar`, siempre que los archivos de `exp_completo` sigan ahí (por ejemplo, guardándolos en Google Drive).

## Notas

- **Soundness:** en ningún árbol terminado se perdió la solución conocida de la instancia (columna `sol. perdidas`).
- **Árboles truncados:** con P grande muchas instancias no terminan en el límite, en parte por los polos de los términos `(suma)^(-1)`, que no se pueden descartar por intervalos. Solo se promedian las instancias que las dos estrategias comparadas terminaron.
- **Diferencias con el paper:** el paper usa Ibex (C++), con ACID y dominios `[-1e8, 1e8]`. Este solver es Python puro y usa dominios `[-2, 2]`, así que la comparación vale por las tendencias, no por los números absolutos.
- Detalles de implementación y decisiones de diseño: `README.md` y `CLAUDE.md` del repositorio.